# Open Judy — Benchmark de Modelos GA, Thinking Levels (`MEDIUM` vs `HIGH`), Prompts & Prompt Optimizer

Este notebook avalia exclusivamente **modelos GA (Generally Available, sem modelos `-preview`)** no Vertex AI sobre os vídeos de audiências judiciais do Open Judy, comparando o impacto de **`thinkingLevel` (`MEDIUM` vs `HIGH`)** e de **múltiplos templates de prompt**, utilizando o **`gemini-4-argon`** como:
1. **Gerador de Ground Truth (Referência Padrão-Ouro em 2 passagens com `thinkingLevel="HIGH"`)**
2. **Juiz Avaliador (LLM-as-a-Judge)**
3. **Meta-Prompt Optimizer (OPRO)** para refinar o prompt do melhor modelo GA e `thinkingLevel`.

### Arquitetura do Experimento (100% Modelos GA)
1. **Modelos GA Avaliados no Vertex AI (`global`)**:
   - `gemini-3.8-flash` (`thinkingLevel`: `MEDIUM` e `HIGH`)
   - `gemini-3.7-flash` (`thinkingLevel`: `MEDIUM` e `HIGH`)
   - `gemini-3.6-flash` (`thinkingLevel`: `MEDIUM` e `HIGH`)
   - `gemini-3.5-flash` (`thinkingLevel`: `MEDIUM` e `HIGH`)
   - `gemini-3.5-flash-lite` (`thinkingLevel`: `MEDIUM` e `HIGH`)
   - `gemini-2.5-flash` (`thinkingLevel`: `MEDIUM` [4096 tokens] e `HIGH` [16384 tokens])
   - `gemini-3.5-transcribe` *(Modelo GA dedicado de transcrição de áudio — `NATIVE_AUDIO`)*
2. **Prompts Base Avaliados**:
   - `P1_CINEMA_SUBTITLES_V2`: Prompt de produção focado em legendas curtas estilo Cinema/Streaming (**4 a 10 segundos**, 1 a 2 linhas curtas) com diarização jurídica e timestamps ancorados à voz real.
   - `P2_VERBATIM_FORENSIC`: Prompt forense *ipsissima verba* focado em literalidade estrita, terminologia processual e identificação detalhada de qualificações.
   - `P3_COMPACT_FAST_DIARIZATION`: Prompt compacto otimizado para baixa latência e menor consumo de tokens.
3. **Prompt Optimizer Automático para o Melhor Modelo GA (`Gemini4ArgonPromptOptimizer`)**:
   - Seleciona automaticamente a combinação vencedora `(Melhor Modelo GA, Melhor Thinking Level)` e executa o ciclo de otimização de prompt contra o Ground Truth do `gemini-4-argon`.

In [1]:
import base64
import csv
import json
import math
import os
import re
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests

# Descobre automaticamente a raiz do repositório Open Judy
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
elif not (REPO_ROOT / "pipelines").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "pipelines").exists():
            REPO_ROOT = parent
            break

OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_KEY_PATHS = [
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS", ""),
    str(REPO_ROOT / "keys" / "service-account.json"),
    str(REPO_ROOT / "services" / ".keys" / "service-account.json"),
    "/keys/service-account.json",
]

CANDIDATE_VIDEO_DIRS = [
    REPO_ROOT / "data" / "hearings",
    REPO_ROOT / "pipelines" / "transcription-pipeline" / ".media" / "hearings",
    Path("/usr/local/google/home/vieiravitor/Documents/open-judy/audiencias"),
]


def resolve_service_account_key() -> Path:
    for candidate in CANDIDATE_KEY_PATHS:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    raise FileNotFoundError(
        f"Service Account JSON key não encontrada. Caminhos verificados: {CANDIDATE_KEY_PATHS}"
    )


def probe_video_duration_seconds(video_path: Path) -> float:
    try:
        res = subprocess.run(
            [
                "ffprobe",
                "-v",
                "error",
                "-show_entries",
                "format=duration",
                "-of",
                "default=noprint_wrappers=1:nokey=1",
                str(video_path),
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=10,
            check=True,
        )
        return round(float(res.stdout.decode("utf-8").strip()), 2)
    except Exception:
        return 0.0


def discover_hearing_videos() -> List[Dict[str, Any]]:
    seen_names = set()
    videos: List[Dict[str, Any]] = []
    for folder in CANDIDATE_VIDEO_DIRS:
        if not folder.is_dir():
            continue
        for f in sorted(folder.iterdir()):
            if f.suffix.lower() in {".mp4", ".webm", ".mkv"} and f.name not in seen_names:
                try:
                    real_f = f.resolve()
                    if not real_f.is_file():
                        continue
                    size_mb = round(real_f.stat().st_size / (1024 * 1024), 2)
                    dur_s = probe_video_duration_seconds(real_f)
                    seen_names.add(f.name)
                    videos.append(
                        {
                            "filename": f.name,
                            "path": real_f,
                            "duration_sec": dur_s,
                            "duration_min": round(dur_s / 60.0, 2),
                            "size_mb": size_mb,
                        }
                    )
                except Exception:
                    continue
    return sorted(videos, key=lambda x: x["duration_sec"])


SA_KEY_PATH = resolve_service_account_key()
AVAILABLE_VIDEOS = discover_hearing_videos()

print(f"Repositório Open Judy : {REPO_ROOT}")
print(f"Service Account Key   : {SA_KEY_PATH}")
print(f"Diretório de Outputs  : {OUTPUT_DIR}")
print(f"\nVídeos de Audiências Encontrados ({len(AVAILABLE_VIDEOS)}):")
print("-" * 88)
for idx, v in enumerate(AVAILABLE_VIDEOS):
    print(
        f"[{idx}] {v['filename']:<28} | Duração: {v['duration_min']:>6.2f} min ({v['duration_sec']:>7.1f}s) | {v['size_mb']:>7.2f} MB"
    )

Repositório Open Judy : /usr/local/google/home/vieiravitor/workspace/work/open-judy
Service Account Key   : /usr/local/google/home/vieiravitor/workspace/work/open-judy/keys/service-account.json
Diretório de Outputs  : /usr/local/google/home/vieiravitor/workspace/work/open-judy/notebooks/outputs

Vídeos de Audiências Encontrados (9):
----------------------------------------------------------------------------------------
[0] 00 - video 1min .mp4         | Duração:   1.47 min (   88.4s) |    6.13 MB
[1] Video 1h 183MB.mp4           | Duração:  84.11 min ( 5046.7s) |  183.17 MB
[2] video 1h30 253MB.mp4         | Duração:  99.91 min ( 5994.7s) |  253.39 MB
[3] 00 - video 2h 194MB  .mp4    | Duração: 130.05 min ( 7803.0s) |  194.59 MB
[4] video 3h 437MB.mp4           | Duração: 193.87 min (11632.2s) |  437.85 MB
[5] video 4h 346 MB.mp4          | Duração: 239.95 min (14396.8s) |  346.87 MB
[6] Video 4h 509MB.mp4           | Duração: 261.47 min (15687.9s) |  509.68 MB
[7] Video 4h  554MB  .m

In [2]:
def _b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")


def format_hms(seconds: float) -> str:
    total_sec = max(0, int(round(seconds)))
    h = total_sec // 3600
    m = (total_sec % 3600) // 60
    s = total_sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}" if h > 0 else f"{m:02d}:{s:02d}"


def parse_timestamp_to_sec(ts: Any) -> float:
    if isinstance(ts, (int, float)):
        return float(ts)
    raw = str(ts or "00:00").strip()
    parts = raw.split(":")
    try:
        if len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        return float(raw)
    except Exception:
        return 0.0


def extract_video_clip_bytes(
    video_path: Path,
    start_sec: float = 0.0,
    duration_sec: float = 120.0,
) -> bytes:
    """
    Extrai e normaliza um recorte de vídeo/áudio via ffmpeg (360p, 2fps, mono 16kHz AAC)
    para envio multimodal consistente a todos os modelos GA no Vertex AI.
    """
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=True) as tmp_out:
        cmd = [
            "ffmpeg",
            "-y",
            "-ss",
            f"{max(0.0, start_sec):.2f}",
            "-i",
            str(video_path),
            "-t",
            f"{max(5.0, duration_sec):.2f}",
            "-vf",
            "scale=-2:360,fps=2",
            "-c:v",
            "libx264",
            "-preset",
            "veryfast",
            "-crf",
            "32",
            "-ac",
            "1",
            "-ar",
            "16000",
            "-c:a",
            "aac",
            "-b:a",
            "32k",
            "-movflags",
            "+faststart",
            tmp_out.name,
        ]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=180)
        return Path(tmp_out.name).read_bytes()


def extract_audio_wav_bytes(
    video_path: Path,
    start_sec: float = 0.0,
    duration_sec: float = 120.0,
) -> bytes:
    """
    Extrai faixa de áudio WAV mono 16kHz PCM para modelos dedicados de áudio como `gemini-3.5-transcribe`.
    """
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as tmp_wav:
        cmd = [
            "ffmpeg",
            "-y",
            "-ss",
            f"{max(0.0, start_sec):.2f}",
            "-i",
            str(video_path),
            "-t",
            f"{max(5.0, duration_sec):.2f}",
            "-vn",
            "-ac",
            "1",
            "-ar",
            "16000",
            "-c:a",
            "pcm_s16le",
            tmp_wav.name,
        ]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=180)
        return Path(tmp_wav.name).read_bytes()


def plain_transcript_to_subtitle_json(raw_text: str, clip_duration_sec: float) -> Dict[str, Any]:
    """
    Converte transcrição contínua (retornada por `gemini-3.5-transcribe` via `audioTranscription`)
    em segmentos proporcionais de legenda (4 a 10s) para comparação justa.
    """
    cleaned = re.sub(r"\s+", " ", raw_text).strip()
    if not cleaned:
        return {"hearingType": "INSTRUCAO_E_JULGAMENTO", "summary": "", "participants": [], "segments": []}

    raw_clauses = [s.strip() for s in re.split(r"(?<=[.?!;:])\s+", cleaned) if s.strip()]
    cues: List[str] = []
    for clause in raw_clauses:
        if len(clause) <= 115:
            cues.append(clause)
        else:
            words = clause.split()
            chunk: List[str] = []
            for w in words:
                chunk.append(w)
                if len(" ".join(chunk)) >= 95:
                    cues.append(" ".join(chunk))
                    chunk = []
            if chunk:
                cues.append(" ".join(chunk))

    total_chars = max(1, sum(len(c) for c in cues))
    cursor = 0.0
    segments = []
    for c in cues:
        dur = max(3.0, (len(c) / total_chars) * clip_duration_sec)
        st = cursor
        et = min(clip_duration_sec, st + dur)
        cursor = et
        segments.append(
            {
                "clipStartTime": format_hms(st),
                "clipEndTime": format_hms(et),
                "speakerRole": "MAGISTRADO",
                "speakerName": "Orador Detectado (Áudio)",
                "text": c,
                "isKeyStatement": False,
                "topicTag": "Transcrição de Áudio",
            }
        )
    if segments:
        segments[-1]["clipEndTime"] = format_hms(clip_duration_sec)

    return {
        "hearingType": "INSTRUCAO_E_JULGAMENTO",
        "summary": cleaned[:220],
        "participants": [{"name": "Orador Detectado (Áudio)", "role": "MAGISTRADO"}],
        "segments": segments,
    }


class VertexAiBenchmarkClient:
    """
    Cliente unificado do Vertex AI (v1beta1) com suporte a `thinkingLevel` ('MEDIUM' e 'HIGH')
    para todos os modelos GA, geração de Ground Truth com `gemini-4-argon` e Prompt Optimization.
    """

    MODEL_PRICING_PER_1M_TOKENS = {
        "gemini-4-argon": {"input": 1.25, "output": 5.00},
        "gemini-3.5-transcribe": {"input": 0.06, "output": 0.24},
        "gemini-3.8-flash": {"input": 0.075, "output": 0.30},
        "gemini-3.7-flash": {"input": 0.075, "output": 0.30},
        "gemini-3.6-flash": {"input": 0.075, "output": 0.30},
        "gemini-3.5-flash": {"input": 0.075, "output": 0.30},
        "gemini-3.5-flash-lite": {"input": 0.0375, "output": 0.15},
        "gemini-2.5-flash": {"input": 0.075, "output": 0.30},
    }

    def __init__(self, key_path: Path, location: str = "global"):
        self.key_path = key_path
        self.location = location
        self._cached_token: Optional[str] = None
        self._token_exp: int = 0
        with open(self.key_path, "r", encoding="utf-8") as f:
            sa = json.load(f)
        self.project_id = sa.get("project_id", "study-deploys")

    def get_access_token(self) -> str:
        now = int(time.time())
        if self._cached_token and now < self._token_exp - 120:
            return self._cached_token

        with open(self.key_path, "r", encoding="utf-8") as f:
            sa = json.load(f)

        token_uri = sa.get("token_uri", "https://oauth2.googleapis.com/token")
        header = _b64url(json.dumps({"alg": "RS256", "typ": "JWT"}).encode("utf-8"))
        claims = _b64url(
            json.dumps(
                {
                    "iss": sa["client_email"],
                    "scope": "https://www.googleapis.com/auth/cloud-platform",
                    "aud": token_uri,
                    "iat": now,
                    "exp": now + 3600,
                }
            ).encode("utf-8")
        )
        signing_input = f"{header}.{claims}".encode("utf-8")

        with tempfile.NamedTemporaryFile("w", suffix=".pem", delete=True) as pem_f:
            pem_f.write(sa["private_key"])
            pem_f.flush()
            res = subprocess.run(
                ["openssl", "dgst", "-sha256", "-sign", pem_f.name],
                input=signing_input,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                check=True,
            )
            sig = _b64url(res.stdout)

        jwt_assertion = f"{header}.{claims}.{sig}"
        resp = requests.post(
            token_uri,
            data={
                "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
                "assertion": jwt_assertion,
            },
            timeout=15,
        )
        resp.raise_for_status()
        tok = resp.json()
        self._cached_token = tok["access_token"]
        self._token_exp = now + int(tok.get("expires_in", 3600))
        return self._cached_token

    @staticmethod
    def clean_json_text(raw: str) -> str:
        text = raw.strip()
        if text.startswith("```"):
            lines = text.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            text = "\n".join(lines).strip()
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return text[start : end + 1]
        return text

    @staticmethod
    def build_thinking_config(model_id: str, thinking_level: Optional[str]) -> Optional[Dict[str, Any]]:
        if not thinking_level or thinking_level == "NATIVE_AUDIO" or "transcribe" in model_id.lower():
            return None
        lvl = thinking_level.upper().strip()
        # `gemini-2.5-flash` usa `thinkingBudget` inteiro equivalente aos níveis MEDIUM (4096) e HIGH (16384)
        if model_id.startswith("gemini-2.5"):
            return {"thinkingBudget": 4096 if lvl == "MEDIUM" else 16384}
        return {"thinkingLevel": lvl}

    def generate_multimodal_json(
        self,
        model_id: str,
        prompt_text: str,
        media_bytes: Optional[bytes] = None,
        audio_wav_bytes: Optional[bytes] = None,
        mime_type: str = "video/mp4",
        system_instruction: Optional[str] = None,
        temperature: float = 0.0,
        max_output_tokens: int = 32768,
        clip_duration_sec: float = 90.0,
        thinking_level: Optional[str] = None,
    ) -> Dict[str, Any]:
        host = (
            "aiplatform.googleapis.com"
            if self.location == "global"
            else f"{self.location}-aiplatform.googleapis.com"
        )
        url = (
            f"https://{host}/v1beta1/projects/{self.project_id}"
            f"/locations/{self.location}/publishers/google/models/{model_id}:generateContent"
        )

        is_transcribe_model = "transcribe" in model_id.lower()
        user_parts: List[Dict[str, Any]] = []
        if is_transcribe_model and audio_wav_bytes is not None:
            user_parts.append(
                {
                    "inlineData": {
                        "mimeType": "audio/wav",
                        "data": base64.b64encode(audio_wav_bytes).decode("ascii"),
                    }
                }
            )
        elif media_bytes is not None:
            user_parts.append(
                {
                    "inlineData": {
                        "mimeType": mime_type,
                        "data": base64.b64encode(media_bytes).decode("ascii"),
                    }
                }
            )
        user_parts.append({"text": prompt_text})

        gen_config: Dict[str, Any] = {
            "temperature": temperature,
            "maxOutputTokens": max_output_tokens,
        }
        if not is_transcribe_model:
            gen_config["responseMimeType"] = "application/json"

        thinking_cfg = self.build_thinking_config(model_id, thinking_level)
        if thinking_cfg is not None:
            gen_config["thinkingConfig"] = thinking_cfg

        payload: Dict[str, Any] = {
            "contents": [{"role": "user", "parts": user_parts}],
            "generationConfig": gen_config,
        }
        if system_instruction and not is_transcribe_model:
            payload["systemInstruction"] = {"parts": [{"text": system_instruction}]}

        last_err = None
        for attempt in range(1, 4):
            t0 = time.perf_counter()
            try:
                token = self.get_access_token()
                resp = requests.post(
                    url,
                    headers={
                        "Authorization": f"Bearer {token}",
                        "Content-Type": "application/json",
                    },
                    json=payload,
                    timeout=240,
                )
                latency_sec = round(time.perf_counter() - t0, 3)
                resp.raise_for_status()
                body = resp.json()

                parts_resp = body.get("candidates", [{}])[0].get("content", {}).get("parts", [])
                raw_text = "".join(
                    (p.get("text") or p.get("audioTranscription", {}).get("text") or "")
                    for p in parts_resp
                    if not p.get("thought")
                )

                cleaned = self.clean_json_text(raw_text)
                try:
                    parsed = json.loads(cleaned)
                except Exception:
                    parsed = plain_transcript_to_subtitle_json(raw_text, clip_duration_sec)

                usage = body.get("usageMetadata", {})
                prompt_tokens = int(usage.get("promptTokenCount", 0))
                output_tokens = int(usage.get("candidatesTokenCount", 0))
                thinking_tokens = int(usage.get("thoughtsTokenCount", 0))
                total_tokens = int(
                    usage.get("totalTokenCount", prompt_tokens + output_tokens + thinking_tokens)
                )

                pricing = self.MODEL_PRICING_PER_1M_TOKENS.get(
                    model_id, {"input": 0.075, "output": 0.30}
                )
                est_cost_usd = round(
                    (prompt_tokens / 1_000_000.0) * pricing["input"]
                    + ((output_tokens + thinking_tokens) / 1_000_000.0) * pricing["output"],
                    6,
                )

                return {
                    "parsed": parsed,
                    "raw_text": raw_text,
                    "latency_sec": latency_sec,
                    "prompt_tokens": prompt_tokens,
                    "output_tokens": output_tokens,
                    "thinking_tokens": thinking_tokens,
                    "total_tokens": total_tokens,
                    "estimated_cost_usd": est_cost_usd,
                }
            except Exception as e:
                last_err = e
                time.sleep(2 * attempt)

        raise RuntimeError(
            f"Falha ao chamar {model_id} (thinking={thinking_level}) após 3 tentativas: {last_err}"
        ) from last_err


vertex_client = VertexAiBenchmarkClient(SA_KEY_PATH)
print(f"Cliente Vertex AI inicializado com sucesso (Projeto: {vertex_client.project_id}).")

Cliente Vertex AI inicializado com sucesso (Projeto: study-deploys).


In [3]:
# ==============================================================================
# 1. PROMPT GERADOR DE GROUND TRUTH (EXCLUSIVO PARA GEMINI 4 ARGON)
# ==============================================================================
GROUND_TRUTH_ARGON_SYSTEM = (
    "Você é o Gemini 4 Argon atuando como Perito Forense Oficial de Transcrição e "
    "Diarização Audiovisual do Poder Judiciário Brasileiro. Sua saída será usada como "
    "Ground Truth (Padrão-Ouro) para avaliar outros modelos e prompts de transcrição."
)

GROUND_TRUTH_ARGON_PROMPT_TEMPLATE = """
Construa o GROUND TRUTH (Referência Padrão-Ouro) de transcrição diarizada e sincronizada para este clipe de audiência judicial brasileira (duração total do clipe: {clip_dur_mmss}, equivalente a {clip_dur_sec:.1f}s, arquivo '{filename}').

REGRAS DO PADRÃO-OURO (GEMINI 4 ARGON):
1. FIDELIDADE LEXICAL E JURÍDICA ABSOLUTA:
   - Transcreva exatamente tudo o que é falado em português no áudio/vídeo do segundo 00:00 até {clip_dur_mmss}, sem omitir perguntas, respostas, qualificações, números de documentos, datas ou intervenções.
2. GRANULARIDADE DE LEGENDAS SINCRONIZADAS (4 A 10 SEGUNDOS):
   - Segmente a fala em blocos curtos de legenda estilo Cinema/Streaming (entre 4 e 10 segundos por segmento, 1 a 2 linhas curtas / 8 a 22 palavras, máximo de 120 caracteres).
   - Ancore `clipStartTime` e `clipEndTime` (formato "MM:SS") exatamente ao segundo em que cada frase começa e termina de ser pronunciada no vídeo.
3. DIARIZAÇÃO FORENSE PRECISA:
   - Identifique cada interlocutor pelo nome real citado ou exibido na tela e classifique `speakerRole` estritamente em:
     `MAGISTRADO`, `PROMOTOR`, `ADVOGADO_AUTOR`, `ADVOGADO_DEFESA`, `REU`, `AUTOR`, `TESTEMUNHA`, `VITIMA` ou `PERITO`.

Retorne APENAS um objeto JSON válido com a estrutura:
{{
  "hearingType": "INSTRUCAO_E_JULGAMENTO",
  "summary": "Síntese técnica do trecho da audiência.",
  "participants": [
    {{"name": "Nome do Participante", "role": "MAGISTRADO"}}
  ],
  "segments": [
    {{
      "clipStartTime": "00:00",
      "clipEndTime": "00:06",
      "speakerRole": "MAGISTRADO",
      "speakerName": "Nome do Orador",
      "text": "Texto exato da fala neste intervalo de 4 a 10 segundos.",
      "isKeyStatement": false,
      "topicTag": "Qualificação"
    }}
  ]
}}
"""

GROUND_TRUTH_AUDIT_PROMPT_TEMPLATE = """
Você é o Gemini 4 Argon realizando a Passagem 2 (Auditoria Forense de Ground Truth) sobre o vídeo anexado (duração: {clip_dur_mmss}) e sobre a transcrição preliminar abaixo.

TRANSCRIÇÃO PRELIMINAR (PASSAGEM 1):
{draft_json}

TAREFA DE AUDITORIA FORENSE:
1. Verifique o áudio/vídeo do início (00:00) ao fim ({clip_dur_mmss}) e corrija qualquer palavra imprecisa, nome próprio grafado incorretamente ou trecho omitido.
2. Verifique se todos os `clipStartTime` e `clipEndTime` ("MM:SS") coincidem exatamente com o instante real da voz no vídeo e se cada segmento tem entre 4 e 10 segundos (1 a 2 linhas curtas de legenda).
3. Verifique se os papéis (`speakerRole`) e nomes (`speakerName`) estão 100% consistentes.
4. Retorne APENAS o JSON final auditado no mesmo formato (`hearingType`, `summary`, `participants`, `segments`).
"""


# ==============================================================================
# 2. REGISTRO DE PROMPTS CANDIDATOS & MODELOS GA (SEM PREVIEW) + THINKING LEVELS
# ==============================================================================
PROMPTS_REGISTRY: Dict[str, Dict[str, str]] = {
    "P1_CINEMA_SUBTITLES_V2": {
        "title": "Prompt Produção v2 (Legendas Cinema 4-10s + Diarização)",
        "description": "Prompt atual do pipeline Open Judy: exige legendas curtas de 4 a 10s (1-2 linhas), timestamps reais da fala e diarização jurídica.",
        "template": """
Você é um sistema especialista em transcrição estenográfica forense e legendagem sincronizada de audiências judiciais brasileiras.
Analise o vídeo anexado (duração: {clip_dur_mmss}, arquivo '{filename}').

REGRAS OBRIGATÓRIAS:
1. GRANULARIDADE DE LEGENDA (4 A 10 SEGUNDOS POR SEGMENTO):
   - Gere segmentos curtos estilo legenda de Cinema/Streaming: cada segmento DEVE durar entre **4 e 10 segundos** e conter **1 a 2 linhas curtas** (8 a 22 palavras, máximo de 120 caracteres).
   - NUNCA agrupe um parágrafo longo em um único segmento de 20 a 60 segundos. Quebre a fala de um mesmo orador a cada frase ou pausa natural.
2. SINCRONIA TEMPORAL EXATA COM A VOZ:
   - `clipStartTime` e `clipEndTime` (formato "MM:SS") DEVEM refletir o segundo exato em que a frase é dita no vídeo entre `00:00` e `{clip_dur_mmss}`.
3. DIARIZAÇÃO E PAPÉIS JURÍDICOS:
   - Use `speakerRole` em: `MAGISTRADO`, `PROMOTOR`, `ADVOGADO_AUTOR`, `ADVOGADO_DEFESA`, `TESTEMUNHA`, `PERITO`, `AUTOR`, `REU`, `VITIMA`.

Retorne APENAS JSON válido:
{{
  "hearingType": "INSTRUCAO_E_JULGAMENTO",
  "summary": "Síntese do trecho.",
  "participants": [{{"name": "Nome", "role": "MAGISTRADO"}}],
  "segments": [
    {{
      "clipStartTime": "00:00",
      "clipEndTime": "00:06",
      "speakerRole": "MAGISTRADO",
      "speakerName": "Nome",
      "text": "Frase curta de legenda (1 a 2 linhas)...",
      "isKeyStatement": false,
      "topicTag": "Tópico"
    }}
  ]
}}
""",
    },
    "P2_VERBATIM_FORENSIC": {
        "title": "Prompt Forense Literal (Ipsissima Verba + Qualificação Estrita)",
        "description": "Enfatiza transcrição verbatim estrita para ata judicial, preservando hesitações relevantes, termos processuais e declarações-chave.",
        "template": """
Atue como Taquígrafo Judicial Sênior de Tribunal de Justiça brasileiro.
Realize a transcrição literal (*ipsissima verba*) e diarizada do vídeo de audiência judicial anexado (duração: {clip_dur_mmss}, arquivo '{filename}').

DIRETRIZES TAQUIGRÁFICAS:
1. Transcreva fielmente cada pergunta e resposta sem resumir e sem parafrasear o depoente, réu, magistrado, promotor ou defesa.
2. Divida os turnos de fala em segmentos de 5 a 12 segundos com timestamps exatos `clipStartTime` e `clipEndTime` ("MM:SS") ancorados ao áudio.
3. Identifique o nome completo ou função de cada participante e classifique `speakerRole` (`MAGISTRADO`, `PROMOTOR`, `ADVOGADO_DEFESA`, `ADVOGADO_AUTOR`, `REU`, `AUTOR`, `TESTEMUNHA`, `VITIMA`, `PERITO`).
4. Marque `isKeyStatement: true` em confissões, negativas de autoria, contradições ou decisões interlocutórias.

Retorne estritamente um objeto JSON com as chaves `hearingType`, `summary`, `participants` e `segments` (`clipStartTime`, `clipEndTime`, `speakerRole`, `speakerName`, `text`, `isKeyStatement`, `topicTag`).
""",
    },
    "P3_COMPACT_FAST_DIARIZATION": {
        "title": "Prompt Compacto (Baixa Latência / Zero-Shot Direto)",
        "description": "Prompt enxuto para avaliar se instruções mínimas reduzem a latência mantendo acurácia aceitável.",
        "template": """
Transcreva em português o vídeo de audiência judicial anexado (00:00 a {clip_dur_mmss}, arquivo '{filename}').
Divida em legendas curtas (4 a 10s) com timestamps reais `clipStartTime` e `clipEndTime` ("MM:SS"), `speakerRole` (`MAGISTRADO`, `PROMOTOR`, `ADVOGADO_DEFESA`, `REU`, `AUTOR`, `TESTEMUNHA`, `VITIMA`), `speakerName` e `text`.
Retorne somente JSON com `hearingType`, `summary`, `participants` e `segments`.
""",
    },
}

# Apenas modelos GA (sem nenhum modelo -preview)
CANDIDATE_MODELS = [
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-3.5-transcribe",
]

THINKING_LEVELS = ["MEDIUM", "HIGH"]

print(f"Ground Truth & Optimizer Model : gemini-4-argon")
print(f"Modelos GA Candidatos ({len(CANDIDATE_MODELS)})      : {CANDIDATE_MODELS}")
print(f"Thinking Levels Testados       : {THINKING_LEVELS} (e NATIVE_AUDIO para gemini-3.5-transcribe)")
print(f"Prompts Candidatos ({len(PROMPTS_REGISTRY)})         : {list(PROMPTS_REGISTRY.keys())}")

Ground Truth & Optimizer Model : gemini-4-argon
Modelos GA Candidatos (7)      : ['gemini-3.8-flash', 'gemini-3.7-flash', 'gemini-3.6-flash', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-2.5-flash', 'gemini-3.5-transcribe']
Thinking Levels Testados       : ['MEDIUM', 'HIGH'] (e NATIVE_AUDIO para gemini-3.5-transcribe)
Prompts Candidatos (3)         : ['P1_CINEMA_SUBTITLES_V2', 'P2_VERBATIM_FORENSIC', 'P3_COMPACT_FAST_DIARIZATION']


In [4]:
# ==============================================================================
# 3. SELEÇÃO DO VÍDEO / TRECHO E GERAÇÃO DO GROUND TRUTH COM GEMINI 4 ARGON
# ==============================================================================
TARGET_VIDEO_INDEX = 0            # 0 -> '00 - video 1min .mp4' (ou altere para 1..8 para outros vídeos)
CLIP_START_SECONDS = 0.0          # Segundo inicial do recorte no vídeo
MAX_CLIP_DURATION_SECONDS = 90.0  # Duração máxima do clipe para o benchmark (ex.: 90s a 300s)
ENABLE_TWO_PASS_ARGON_AUDIT = True
FORCE_REGENERATE_GT = False

selected_video = AVAILABLE_VIDEOS[TARGET_VIDEO_INDEX]
clip_duration_sec = min(
    MAX_CLIP_DURATION_SECONDS,
    max(10.0, selected_video["duration_sec"] - CLIP_START_SECONDS),
)
clip_dur_mmss = format_hms(clip_duration_sec)

print(
    f"Extraindo clipe multimodal (MP4) e faixa de áudio (WAV) de '{selected_video['filename']}' "
    f"({format_hms(CLIP_START_SECONDS)} -> {format_hms(CLIP_START_SECONDS + clip_duration_sec)}, {clip_duration_sec:.1f}s)..."
)
clip_bytes = extract_video_clip_bytes(
    selected_video["path"],
    start_sec=CLIP_START_SECONDS,
    duration_sec=clip_duration_sec,
)
audio_wav_bytes = extract_audio_wav_bytes(
    selected_video["path"],
    start_sec=CLIP_START_SECONDS,
    duration_sec=clip_duration_sec,
)
print(
    f"Clipe MP4 extraído: {len(clip_bytes) / 1024:.1f} KB | "
    f"Faixa WAV (para gemini-3.5-transcribe): {len(audio_wav_bytes) / 1024:.1f} KB"
)


def normalize_segments_list(raw_segments: List[Dict[str, Any]], clip_dur: float) -> List[Dict[str, Any]]:
    normalized = []
    for idx, seg in enumerate(raw_segments, start=1):
        st = max(0.0, min(clip_dur, parse_timestamp_to_sec(seg.get("clipStartTime") or seg.get("startTime") or 0)))
        et = max(st + 1.0, min(clip_dur, parse_timestamp_to_sec(seg.get("clipEndTime") or seg.get("endTime") or (st + 5.0))))
        text = str(seg.get("text") or "").strip()
        if not text:
            continue
        normalized.append(
            {
                "index": idx,
                "startSec": round(st, 2),
                "endSec": round(et, 2),
                "durationSec": round(et - st, 2),
                "startTimeFormatted": format_hms(st),
                "endTimeFormatted": format_hms(et),
                "speakerRole": str(seg.get("speakerRole") or "PARTICIPANTE").upper().strip(),
                "speakerName": str(seg.get("speakerName") or "Participante").strip(),
                "text": text,
                "charLength": len(text),
                "wordCount": len(text.split()),
                "isKeyStatement": bool(seg.get("isKeyStatement", False)),
                "topicTag": str(seg.get("topicTag") or "Depoimento"),
            }
        )
    return normalized


gt_cache_file = (
    OUTPUT_DIR
    / f"ground_truth_argon_{re.sub(r'[^a-zA-Z0-9_-]', '_', selected_video['filename'])}_{int(CLIP_START_SECONDS)}s_{int(clip_duration_sec)}s.json"
)

if gt_cache_file.is_file() and not FORCE_REGENERATE_GT:
    print(f"\nCarregando Ground Truth (Gemini 4 Argon) do cache: {gt_cache_file.name}")
    ground_truth_record = json.loads(gt_cache_file.read_text(encoding="utf-8"))
else:
    print("\n[Passagem 1/2] Gerando Ground Truth inicial com `gemini-4-argon`...")
    p1_prompt = GROUND_TRUTH_ARGON_PROMPT_TEMPLATE.format(
        clip_dur_mmss=clip_dur_mmss,
        clip_dur_sec=clip_duration_sec,
        filename=selected_video["filename"],
    )
    res_p1 = vertex_client.generate_multimodal_json(
        model_id="gemini-4-argon",
        prompt_text=p1_prompt,
        media_bytes=clip_bytes,
        system_instruction=GROUND_TRUTH_ARGON_SYSTEM,
        temperature=0.0,
        clip_duration_sec=clip_duration_sec,
    )
    gt_parsed = res_p1["parsed"]
    total_gt_latency = res_p1["latency_sec"]
    total_gt_tokens = res_p1["total_tokens"]

    if ENABLE_TWO_PASS_ARGON_AUDIT:
        print("[Passagem 2/2] Executando Auditoria Forense de Timestamps & Diarização com `gemini-4-argon`...")
        p2_prompt = GROUND_TRUTH_AUDIT_PROMPT_TEMPLATE.format(
            clip_dur_mmss=clip_dur_mmss,
            draft_json=json.dumps(gt_parsed, ensure_ascii=False, indent=2),
        )
        res_p2 = vertex_client.generate_multimodal_json(
            model_id="gemini-4-argon",
            prompt_text=p2_prompt,
            media_bytes=clip_bytes,
            system_instruction=GROUND_TRUTH_ARGON_SYSTEM,
            temperature=0.0,
            clip_duration_sec=clip_duration_sec,
        )
        gt_parsed = res_p2["parsed"]
        total_gt_latency = round(total_gt_latency + res_p2["latency_sec"], 3)
        total_gt_tokens += res_p2["total_tokens"]

    gt_segments = normalize_segments_list(gt_parsed.get("segments", []), clip_duration_sec)
    ground_truth_record = {
        "model": "gemini-4-argon",
        "video_filename": selected_video["filename"],
        "clip_start_sec": CLIP_START_SECONDS,
        "clip_duration_sec": clip_duration_sec,
        "two_pass_audit": ENABLE_TWO_PASS_ARGON_AUDIT,
        "latency_sec": total_gt_latency,
        "total_tokens": total_gt_tokens,
        "summary": gt_parsed.get("summary", ""),
        "participants": gt_parsed.get("participants", []),
        "segments": gt_segments,
        "full_text": " ".join(s["text"] for s in gt_segments),
    }
    gt_cache_file.write_text(json.dumps(ground_truth_record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Ground Truth salvo em: {gt_cache_file}")

print("\n" + "=" * 95)
print(
    f"GROUND TRUTH (GEMINI 4 ARGON) — {ground_truth_record['video_filename']} "
    f"({len(ground_truth_record['segments'])} legendas | {ground_truth_record['latency_sec']}s)"
)
print("=" * 95)
for seg in ground_truth_record["segments"][:15]:
    print(
        f"[{seg['startTimeFormatted']} - {seg['endTimeFormatted']}] "
        f"({seg['durationSec']:>4.1f}s) {seg['speakerRole']:<15} | {seg['speakerName']}: {seg['text']}"
    )
if len(ground_truth_record["segments"]) > 15:
    print(f"... (+{len(ground_truth_record['segments']) - 15} segmentos no Ground Truth)")

Extraindo clipe multimodal (MP4) e faixa de áudio (WAV) de '00 - video 1min .mp4' (00:00 -> 01:28, 88.4s)...
Clipe MP4 extraído: 786.3 KB | Faixa WAV (para gemini-3.5-transcribe): 2757.1 KB

Carregando Ground Truth (Gemini 4 Argon) do cache: ground_truth_argon_00_-_video_1min__mp4_0s_88s.json

GROUND TRUTH (GEMINI 4 ARGON) — 00 - video 1min .mp4 (15 legendas | 65.051s)
[00:03 - 00:08] ( 5.0s) MAGISTRADO      | Conciliadora CEJUSC Muriaé: É, vamos lá. As partes celebraram um acordo nos seguintes termos:
[00:08 - 00:16] ( 8.0s) MAGISTRADO      | Conciliadora CEJUSC Muriaé: Um, do reconhecimento da união estável. As partes informaram que se uniram em 2020, conforme consta nos autos,
[00:16 - 00:22] ( 6.0s) MAGISTRADO      | Conciliadora CEJUSC Muriaé: é, comprovante de residência e fotos com a reclamante e sua família, e ambos requerem
[00:22 - 00:27] ( 5.0s) MAGISTRADO      | Conciliadora CEJUSC Muriaé: requerem que seja declarada a união estável para os devidos fins.
[00:28 - 00:35] ( 7

In [5]:
# ==============================================================================
# 4. MOTOR DE MÉTRICAS (WER, CER, DIARIZAÇÃO, TIMESTAMPS & JUIZ GEMINI 4 ARGON)
# ==============================================================================
def normalize_pt_text(text: str) -> str:
    t = text.lower().strip()
    t = re.sub(r"[^\w\sáàâãéêíóôõúüç]", " ", t, flags=re.UNICODE)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def levenshtein_distance(seq_a: List[Any], seq_b: List[Any]) -> int:
    n, m = len(seq_a), len(seq_b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    curr = [0] * (m + 1)
    for i in range(1, n + 1):
        curr[0] = i
        a_i = seq_a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if a_i == seq_b[j - 1] else 1
            curr[j] = min(
                prev[j] + 1,       # deleção
                curr[j - 1] + 1,   # inserção
                prev[j - 1] + cost # substituição
            )
        prev, curr = curr, prev
    return prev[m]


def compute_wer_and_cer(gt_text: str, hyp_text: str) -> Tuple[float, float]:
    gt_norm = normalize_pt_text(gt_text)
    hyp_norm = normalize_pt_text(hyp_text)

    gt_words = gt_norm.split()
    hyp_words = hyp_norm.split()
    wer = (
        levenshtein_distance(gt_words, hyp_words) / max(1, len(gt_words))
        if gt_words
        else 0.0
    )

    gt_chars = list(gt_norm.replace(" ", ""))
    hyp_chars = list(hyp_norm.replace(" ", ""))
    cer = (
        levenshtein_distance(gt_chars, hyp_chars) / max(1, len(gt_chars))
        if gt_chars
        else 0.0
    )
    return round(wer * 100.0, 2), round(cer * 100.0, 2)


def compute_temporal_and_diarization_metrics(
    gt_segments: List[Dict[str, Any]],
    hyp_segments: List[Dict[str, Any]],
) -> Dict[str, float]:
    """
    Alinha cada segmento do Ground Truth ao segmento candidato com maior sobreposição
    ou menor distância de ponto médio para calcular:
      - Desvio médio de timestamp em segundos (mean_timestamp_drift_sec)
      - Acurácia de papel jurídico (speaker_role_accuracy_pct)
      - Conformidade de legenda estilo cinema (4-10s e <= 125 chars)
    """
    if not hyp_segments:
        return {
            "mean_timestamp_drift_sec": 99.0,
            "speaker_role_accuracy_pct": 0.0,
            "subtitle_compliance_pct": 0.0,
            "avg_segment_duration_sec": 0.0,
            "avg_segment_chars": 0.0,
        }

    drifts: List[float] = []
    role_matches = 0

    for gt in gt_segments:
        gt_mid = 0.5 * (gt["startSec"] + gt["endSec"])
        # Busca o segmento candidato mais próximo temporalmente
        best_hyp = min(
            hyp_segments,
            key=lambda h: abs(0.5 * (h["startSec"] + h["endSec"]) - gt_mid),
        )
        start_err = abs(best_hyp["startSec"] - gt["startSec"])
        end_err = abs(best_hyp["endSec"] - gt["endSec"])
        drifts.append(0.5 * (start_err + end_err))

        gt_role = gt["speakerRole"]
        hyp_role = best_hyp["speakerRole"]
        if (
            gt_role == hyp_role
            or (gt_role in hyp_role)
            or (hyp_role in gt_role)
            or ({gt_role, hyp_role} <= {"REU", "TESTEMUNHA", "DEPOENTE", "INTERROGADO"})
        ):
            role_matches += 1

    compliant_count = sum(
        1
        for h in hyp_segments
        if 3.0 <= h["durationSec"] <= 11.5 and h["charLength"] <= 125
    )

    return {
        "mean_timestamp_drift_sec": round(sum(drifts) / max(1, len(drifts)), 2),
        "speaker_role_accuracy_pct": round((role_matches / max(1, len(gt_segments))) * 100.0, 1),
        "subtitle_compliance_pct": round((compliant_count / max(1, len(hyp_segments))) * 100.0, 1),
        "avg_segment_duration_sec": round(
            sum(h["durationSec"] for h in hyp_segments) / max(1, len(hyp_segments)), 2
        ),
        "avg_segment_chars": round(
            sum(h["charLength"] for h in hyp_segments) / max(1, len(hyp_segments)), 1
        ),
    }


def evaluate_with_argon_judge(
    client: VertexAiBenchmarkClient,
    gt_record: Dict[str, Any],
    candidate_segments: List[Dict[str, Any]],
    candidate_model: str,
    candidate_prompt_key: str,
) -> Dict[str, Any]:
    """
    Usa o `gemini-4-argon` como Juiz Forense (LLM-as-a-Judge) comparando a saída do
    par (Modelo, Prompt) contra o Ground Truth de referência.
    """
    judge_prompt = f"""
Você é o Gemini 4 Argon atuando como Juiz Avaliador de Qualidade de Transcrição Judicial.
Compare a TRANSCRIÇÃO CANDIDATA abaixo (gerada por modelo='{candidate_model}', prompt='{candidate_prompt_key}')
contra o GROUND TRUTH OFICIAL da audiência.

GROUND TRUTH OFICIAL (GEMINI 4 ARGON):
{json.dumps(gt_record['segments'], ensure_ascii=False)}

TRANSCRIÇÃO CANDIDATA AVALIADA:
{json.dumps(candidate_segments, ensure_ascii=False)}

Avalie de 0 a 100 os seguintes critérios técnicos:
1. `factual_fidelity_score` (0-100): Fidelidade fática, ausência de alucinações ou omissões de perguntas/respostas.
2. `legal_terminology_score` (0-100): Correção de nomes próprios, termos jurídicos e números citados.
3. `diarization_score` (0-100): Precisão na separação de quem fala e atribuição correta do papel processual.
4. `subtitle_ux_score` (0-100): Qualidade da quebra em legendas curtas sincronizadas (4 a 10s, 1-2 linhas legíveis).
5. `overall_judge_score` (0-100): Nota global ponderada.

Retorne APENAS JSON válido:
{{
  "factual_fidelity_score": 95,
  "legal_terminology_score": 94,
  "diarization_score": 92,
  "subtitle_ux_score": 96,
  "overall_judge_score": 94,
  "strengths": "Resumo curto dos pontos fortes em português.",
  "weaknesses": "Resumo curto das falhas ou desvios observados em português."
}}
"""
    try:
        res = client.generate_multimodal_json(
            model_id="gemini-4-argon",
            prompt_text=judge_prompt,
            media_bytes=None,
            system_instruction=GROUND_TRUTH_ARGON_SYSTEM,
            temperature=0.0,
        )
        return res["parsed"]
    except Exception as e:
        return {
            "factual_fidelity_score": 0,
            "legal_terminology_score": 0,
            "diarization_score": 0,
            "subtitle_ux_score": 0,
            "overall_judge_score": 0,
            "strengths": "",
            "weaknesses": f"Erro na avaliação do juiz: {e}",
        }

In [6]:
# ==============================================================================
# 5. EXECUÇÃO PARALELA DA MATRIZ DE BENCHMARK (MODELOS GA × THINKING LEVEL × PROMPTS)
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed


def render_prompt_template(template_str: str, clip_dur_mmss: str, clip_dur_sec: float, filename: str) -> str:
    return (
        template_str.replace("{{clip_dur_mmss}}", clip_dur_mmss)
        .replace("{clip_dur_mmss}", clip_dur_mmss)
        .replace("{{clip_dur_sec}}", f"{clip_dur_sec:.1f}")
        .replace("{clip_dur_sec}", f"{clip_dur_sec:.1f}")
        .replace("{{filename}}", filename)
        .replace("{filename}", filename)
        .replace("{{", "{")
        .replace("}}", "}")
    )


def evaluate_single_model_prompt_run(
    model_id: str,
    thinking_level: str,
    prompt_key: str,
    prompt_title: str,
    prompt_template: str,
) -> Dict[str, Any]:
    rendered_prompt = render_prompt_template(
        prompt_template,
        clip_dur_mmss=clip_dur_mmss,
        clip_dur_sec=clip_duration_sec,
        filename=selected_video["filename"],
    )
    call_res = vertex_client.generate_multimodal_json(
        model_id=model_id,
        prompt_text=rendered_prompt,
        media_bytes=clip_bytes,
        audio_wav_bytes=audio_wav_bytes,
        temperature=0.0,
        clip_duration_sec=clip_duration_sec,
        thinking_level=thinking_level,
    )
    parsed = call_res["parsed"]
    hyp_segments = normalize_segments_list(parsed.get("segments", []), clip_duration_sec)
    hyp_full_text = " ".join(s["text"] for s in hyp_segments)

    wer_pct, cer_pct = compute_wer_and_cer(ground_truth_record["full_text"], hyp_full_text)
    td_metrics = compute_temporal_and_diarization_metrics(
        ground_truth_record["segments"], hyp_segments
    )
    judge_scores = evaluate_with_argon_judge(
        vertex_client,
        ground_truth_record,
        hyp_segments,
        f"{model_id} (thinking={thinking_level})",
        prompt_key,
    )

    lexical_accuracy = max(0.0, 100.0 - wer_pct)
    composite_score = round(
        0.35 * float(judge_scores.get("overall_judge_score", 0))
        + 0.30 * lexical_accuracy
        + 0.20 * td_metrics["speaker_role_accuracy_pct"]
        + 0.15 * td_metrics["subtitle_compliance_pct"],
        2,
    )
    rtf = round(call_res["latency_sec"] / max(1.0, clip_duration_sec), 3)

    return {
        "model_id": model_id,
        "thinking_level": thinking_level,
        "prompt_key": prompt_key,
        "prompt_title": prompt_title,
        "video_filename": selected_video["filename"],
        "clip_duration_sec": clip_duration_sec,
        "composite_score": composite_score,
        "argon_judge_score": float(judge_scores.get("overall_judge_score", 0)),
        "factual_fidelity_score": float(judge_scores.get("factual_fidelity_score", 0)),
        "legal_terminology_score": float(judge_scores.get("legal_terminology_score", 0)),
        "wer_pct": wer_pct,
        "cer_pct": cer_pct,
        "speaker_role_accuracy_pct": td_metrics["speaker_role_accuracy_pct"],
        "mean_timestamp_drift_sec": td_metrics["mean_timestamp_drift_sec"],
        "subtitle_compliance_pct": td_metrics["subtitle_compliance_pct"],
        "segment_count": len(hyp_segments),
        "avg_segment_duration_sec": td_metrics["avg_segment_duration_sec"],
        "avg_segment_chars": td_metrics["avg_segment_chars"],
        "latency_sec": call_res["latency_sec"],
        "rtf": rtf,
        "prompt_tokens": call_res["prompt_tokens"],
        "output_tokens": call_res["output_tokens"],
        "thinking_tokens": call_res.get("thinking_tokens", 0),
        "estimated_cost_usd": call_res["estimated_cost_usd"],
        "strengths": judge_scores.get("strengths", ""),
        "weaknesses": judge_scores.get("weaknesses", ""),
        "segments": hyp_segments,
    }


BENCHMARK_RESULTS: List[Dict[str, Any]] = []
tasks_matrix = []
for m in CANDIDATE_MODELS:
    m_levels = ["NATIVE_AUDIO"] if "transcribe" in m.lower() else THINKING_LEVELS
    for t_lvl in m_levels:
        for pk, pm in PROMPTS_REGISTRY.items():
            tasks_matrix.append((m, t_lvl, pk, pm["title"], pm["template"]))

total_runs = len(tasks_matrix)

print(
    f"Iniciando matriz de benchmark de modelos GA em paralelo ({total_runs} execuções: "
    f"Modelos GA × Thinking Levels [MEDIUM, HIGH] × 3 Prompts) sobre '{selected_video['filename']}' ({clip_dur_mmss})...\n"
)

completed_count = 0
with ThreadPoolExecutor(max_workers=8) as pool:
    future_map = {
        pool.submit(evaluate_single_model_prompt_run, m, t_lvl, pk, title, tmpl): (m, t_lvl, pk)
        for (m, t_lvl, pk, title, tmpl) in tasks_matrix
    }
    for fut in as_completed(future_map):
        completed_count += 1
        m, t_lvl, pk = future_map[fut]
        try:
            row = fut.result()
            BENCHMARK_RESULTS.append(row)
            print(
                f"[{completed_count:02d}/{total_runs:02d}] {m:<21} | Think:{t_lvl:<12} | {pk:<27} -> "
                f"Score: {row['composite_score']:5.1f} | Argon: {row['argon_judge_score']:4.0f} | "
                f"WER: {row['wer_pct']:5.1f}% | CER: {row['cer_pct']:5.1f}% | Diar: {row['speaker_role_accuracy_pct']:5.1f}% | "
                f"Leg 4-10s: {row['subtitle_compliance_pct']:5.1f}% | ThinkTok: {row['thinking_tokens']:>5} | Tempo: {row['latency_sec']:5.2f}s"
            )
        except Exception as err:
            print(f"[{completed_count:02d}/{total_runs:02d}] [ERRO] {m} ({t_lvl}) + {pk}: {err}")

BENCHMARK_RESULTS.sort(key=lambda r: (-r["composite_score"], r["wer_pct"], r["latency_sec"]))
print(f"\nMatriz GA concluída com {len(BENCHMARK_RESULTS)} execuções avaliadas.")

Iniciando matriz de benchmark de modelos GA em paralelo (39 execuções: Modelos GA × Thinking Levels [MEDIUM, HIGH] × 3 Prompts) sobre '00 - video 1min .mp4' (01:28)...

[01/39] gemini-3.7-flash      | Think:MEDIUM       | P1_CINEMA_SUBTITLES_V2      -> Score:  89.3 | Argon:   95 | WER:   8.8% | CER:   6.6% | Diar: 100.0% | Leg 4-10s:  58.3% | ThinkTok:   725 | Tempo: 11.65s
[02/39] gemini-3.8-flash      | Think:MEDIUM       | P2_VERBATIM_FORENSIC        -> Score:  83.6 | Argon:   94 | WER:   3.3% | CER:   1.4% | Diar:  93.3% | Leg 4-10s:  20.0% | ThinkTok:   774 | Tempo: 13.09s
[03/39] gemini-3.7-flash      | Think:MEDIUM       | P2_VERBATIM_FORENSIC        -> Score:  82.7 | Argon:   93 | WER:   5.0% | CER:   1.9% | Diar: 100.0% | Leg 4-10s:  11.1% | ThinkTok:   582 | Tempo: 16.13s
[04/39] gemini-3.8-flash      | Think:HIGH         | P2_VERBATIM_FORENSIC        -> Score:  85.3 | Argon:   93 | WER:   4.4% | CER:   1.7% | Diar:  93.3% | Leg 4-10s:  36.4% | ThinkTok:  2128 | Tempo: 22.56s

In [7]:
# ==============================================================================
# 6. LEADERBOARD GA, ANÁLISE THINKING LEVEL (MEDIUM vs HIGH) & DIFF LADO A LADO
# ==============================================================================
print("=" * 156)
print(
    f"{'RANK':<5} {'MODELO GA':<21} {'THINKING':<13} {'PROMPT':<28} {'SCORE':>7} {'ARGON':>7} "
    f"{'WER%':>7} {'CER%':>7} {'DIAR%':>7} {'DRIFT(s)':>9} {'LEG 4-10s':>10} {'THINK_TOK':>10} {'LAT(s)':>8} {'CUSTO($)':>10}"
)
print("=" * 156)
for rank, r in enumerate(BENCHMARK_RESULTS, start=1):
    print(
        f"#{rank:<4} {r['model_id']:<21} {r.get('thinking_level', 'N/A'):<13} {r['prompt_key']:<28} "
        f"{r['composite_score']:>7.2f} {r['argon_judge_score']:>7.1f} "
        f"{r['wer_pct']:>6.1f}% {r['cer_pct']:>6.1f}% {r['speaker_role_accuracy_pct']:>6.1f}% "
        f"{r['mean_timestamp_drift_sec']:>8.2f}s {r['subtitle_compliance_pct']:>9.1f}% "
        f"{r.get('thinking_tokens', 0):>10} {r['latency_sec']:>7.2f}s ${r['estimated_cost_usd']:>8.5f}"
    )
print("=" * 156)

# Tabela resumo comparando MEDIUM vs HIGH por modelo GA
print("\n" + "=" * 120)
print("COMPARATIVO DE IMPACTO DO THINKING LEVEL (MEDIUM vs HIGH) POR MODELO GA (MÉDIA DOS 3 PROMPTS)")
print("=" * 120)
print(
    f"{'MODELO GA':<22} {'THINKING LEVEL':<15} {'MÉDIA SCORE':>12} {'MELHOR SCORE':>13} "
    f"{'MÉDIA WER%':>11} {'MÉDIA DIAR%':>12} {'MÉDIA THINK_TOK':>16} {'MÉDIA LAT(s)':>13}"
)
print("-" * 120)
for m in CANDIDATE_MODELS:
    m_levels = ["NATIVE_AUDIO"] if "transcribe" in m.lower() else THINKING_LEVELS
    for t_lvl in m_levels:
        sub = [
            r for r in BENCHMARK_RESULTS
            if r["model_id"] == m and r.get("thinking_level") == t_lvl and not r.get("is_optimized")
        ]
        if not sub:
            continue
        avg_score = sum(x["composite_score"] for x in sub) / len(sub)
        max_score = max(x["composite_score"] for x in sub)
        avg_wer = sum(x["wer_pct"] for x in sub) / len(sub)
        avg_diar = sum(x["speaker_role_accuracy_pct"] for x in sub) / len(sub)
        avg_ttok = int(round(sum(x.get("thinking_tokens", 0) for x in sub) / len(sub)))
        avg_lat = sum(x["latency_sec"] for x in sub) / len(sub)
        print(
            f"{m:<22} {t_lvl:<15} {avg_score:>12.2f} {max_score:>13.2f} "
            f"{avg_wer:>10.1f}% {avg_diar:>11.1f}% {avg_ttok:>16} {avg_lat:>12.2f}s"
        )
print("=" * 120)

if BENCHMARK_RESULTS:
    best = BENCHMARK_RESULTS[0]
    print(
        f"\nMELHOR COMBINAÇÃO GA: {best['model_id']} (thinkingLevel={best.get('thinking_level')}) "
        f"+ {best['prompt_key']} (Score: {best['composite_score']})"
    )
    print(f"  + Pontos Fortes (Juiz Argon): {best['strengths']}")
    print(f"  - Pontos de Atenção         : {best['weaknesses']}")

    print("\n" + "=" * 125)
    print("COMPARAÇÃO LADO A LADO POR JANELA TEMPORAL (GROUND TRUTH GEMINI 4 ARGON vs TOP 3 MODELOS GA)")
    print("=" * 125)
    top_candidates = BENCHMARK_RESULTS[:3]
    for gt_seg in ground_truth_record["segments"][:8]:
        gt_mid = 0.5 * (gt_seg["startSec"] + gt_seg["endSec"])
        print(
            f"\n[GT gemini-4-argon] ({gt_seg['startTimeFormatted']} - {gt_seg['endTimeFormatted']}) "
            f"[{gt_seg['speakerRole']} | {gt_seg['speakerName']}]: {gt_seg['text']}"
        )
        for cand in top_candidates:
            if not cand["segments"]:
                continue
            matched = min(
                cand["segments"],
                key=lambda h: abs(0.5 * (h["startSec"] + h["endSec"]) - gt_mid),
            )
            print(
                f"  ├─ {cand['model_id']} [{cand.get('thinking_level')}] ({cand['prompt_key']}) "
                f"({matched['startTimeFormatted']} - {matched['endTimeFormatted']}) "
                f"[{matched['speakerRole']} | {matched['speakerName']}]: {matched['text']}"
            )

RANK  MODELO GA             THINKING      PROMPT                         SCORE   ARGON    WER%    CER%   DIAR%  DRIFT(s)  LEG 4-10s  THINK_TOK   LAT(s)   CUSTO($)
#1    gemini-2.5-flash      HIGH          P2_VERBATIM_FORENSIC           96.23    96.0    1.7%    0.5%  100.0%     1.37s      87.5%       2252   19.94s $ 0.00313
#2    gemini-3.7-flash      HIGH          P1_CINEMA_SUBTITLES_V2         95.38    98.0    7.2%    4.8%  100.0%     0.73s      88.2%       1584   18.54s $ 0.00168
#3    gemini-2.5-flash      MEDIUM        P3_COMPACT_FAST_DIARIZATION    95.30    96.0    2.2%    1.6%  100.0%     1.37s      82.4%        846   13.26s $ 0.00259
#4    gemini-3.5-flash      MEDIUM        P1_CINEMA_SUBTITLES_V2         94.51    98.0    7.2%    3.6%  100.0%     0.90s      82.4%       2098   22.41s $ 0.00184
#5    gemini-3.8-flash      HIGH          P1_CINEMA_SUBTITLES_V2         94.49    98.0    6.6%    4.8%  100.0%     0.93s      81.2%       1754   46.36s $ 0.00170
#6    gemini-3.5-flash     

In [8]:
# ==============================================================================
# 7. PROMPT OPTIMIZER AUTOMÁTICO (GEMINI 4 ARGON META-OPTIMIZER) PARA O MELHOR MODELO GA
# ==============================================================================
def optimize_prompt_for_winning_model(
    client: VertexAiBenchmarkClient,
    gt_record: Dict[str, Any],
    best_run: Dict[str, Any],
    base_prompt_template: str,
) -> Dict[str, Any]:
    """
    Usa o `gemini-4-argon` como Meta-Prompt Optimizer (estilo OPRO / Automatic Prompt Engineering):
    analisa os erros lexicais, desvios de timestamp e falhas de diarização do modelo GA vencedor
    contra o Ground Truth e sintetiza 2 prompts concisos e cirurgicamente otimizados para esse modelo.
    """
    optimizer_instruction = f"""
Você é o Gemini 4 Argon atuando como Engenheiro de Meta-Prompt Optimization (OPRO) para transcrição judicial multimodal.

O modelo GA vencedor do benchmark inicial foi `{best_run['model_id']}` (com `thinkingLevel={best_run.get('thinking_level')}`) usando o prompt `{best_run['prompt_key']}`.
MÉTRICAS ATUAIS DO MODELO GA VENCEDOR:
- Composite Score: {best_run['composite_score']} / 100
- Nota do Juiz Argon: {best_run['argon_judge_score']} / 100
- Word Error Rate (WER): {best_run['wer_pct']}%
- Character Error Rate (CER): {best_run['cer_pct']}%
- Acurácia de Diarização (Speaker Role): {best_run['speaker_role_accuracy_pct']}%
- Drift Médio de Timestamp: {best_run['mean_timestamp_drift_sec']}s
- Conformidade de Legendas Curtas (4-10s, <=125 chars): {best_run['subtitle_compliance_pct']}%
- Pontos Fortes: {best_run['strengths']}
- Falhas / Pontos Fracos Detectados: {best_run['weaknesses']}

PROMPT BASE ORIGINAL:
{base_prompt_template}

AMOSTRA DO GROUND TRUTH OFICIAL (GEMINI 4 ARGON):
{json.dumps(gt_record['segments'], ensure_ascii=False, indent=2)}

AMOSTRA DA SAÍDA ATUAL DO MODELO `{best_run['model_id']}`:
{json.dumps(best_run['segments'], ensure_ascii=False, indent=2)}

SUA TAREFA DE OTIMIZAÇÃO DE PROMPT:
Gere 2 novos templates de prompt cirurgicamente otimizados para o modelo `{best_run['model_id']}` (`thinkingLevel={best_run.get('thinking_level')}`):
1. REGRA DE CONCISÃO (CRÍTICO): Mantenha cada prompt otimizado CONCISO (entre 900 e 1400 caracteres no máximo) para evitar truncamento do JSON.
2. Preservar os pontos fortes do PROMPT BASE ORIGINAL, adicionando cirurgicamente:
   - Cada segmento deve ter entre **4 e 10 segundos** (1 a 2 linhas curtas, até 115 caracteres), com exceção de confirmações rápidas/monossilábicas ("Sim.") no final, que devem ter sua duração real (1 a 3 segundos).
   - Proibição de agrupar falas de dois locutores no mesmo segmento: quando o reclamado (`REU`) e a reclamante (`AUTOR`) responderem "Sim" no final, emitir dois objetos separados no array `segments` (`"speakerRole": "REU"` e `"speakerRole": "AUTOR"`).
   - Preservação literal de interjeições iniciais ("É, vamos lá.", "é, comprovante") e preposições ("gratuidade da justiça") sem adicionar reticências "...".
3. Use os placeholders `{{clip_dur_mmss}}` e `{{filename}}` e inclua o bloco de exemplo JSON final com chaves normais.

Retorne APENAS JSON válido no formato:
{{
  "diagnosis": "Diagnóstico técnico detalhado das causas de erro do prompt anterior.",
  "optimized_prompts": [
    {{
      "key": "OPT_V1_DIARIZATION_SURGICAL",
      "title": "Prompt Otimizado v1 (Cirúrgico: Isolamento REU/AUTOR + Verbatim + 4-10s)",
      "rationale": "Explicação das melhorias aplicadas neste prompt.",
      "template": "Texto conciso do prompt otimizado..."
    }},
    {{
      "key": "OPT_V2_PROSODY_ALIGNED",
      "title": "Prompt Otimizado v2 (Alinhamento Prosódico & Turnos Individuais)",
      "rationale": "Explicação das melhorias aplicadas neste prompt.",
      "template": "Texto conciso do segundo prompt otimizado..."
    }}
  ]
}}
"""
    res = client.generate_multimodal_json(
        model_id="gemini-4-argon",
        prompt_text=optimizer_instruction,
        media_bytes=None,
        system_instruction=GROUND_TRUTH_ARGON_SYSTEM,
        temperature=0.1,
        max_output_tokens=32768,
        thinking_level="HIGH",
    )
    return res["parsed"]


OPTIMIZATION_REPORT: Dict[str, Any] = {}

if BENCHMARK_RESULTS:
    base_results_only = [r for r in BENCHMARK_RESULTS if not r.get("is_optimized")]
    base_results_only.sort(key=lambda r: (-r["composite_score"], r["wer_pct"], r["latency_sec"]))
    initial_winner = base_results_only[0]
    best_model_id = initial_winner["model_id"]
    best_thinking_lvl = initial_winner.get("thinking_level", "HIGH")
    base_prompt_key = initial_winner["prompt_key"]
    base_prompt_template = PROMPTS_REGISTRY[base_prompt_key]["template"]

    print("=" * 118)
    print(
        f"INICIANDO PROMPT OPTIMIZER (GEMINI 4 ARGON) PARA O MODELO GA VENCEDOR: {best_model_id} "
        f"[thinkingLevel={best_thinking_lvl}] (Prompt Base: {base_prompt_key} | Score Inicial: {initial_winner['composite_score']})"
    )
    print("=" * 118)

    opt_meta = optimize_prompt_for_winning_model(
        vertex_client,
        ground_truth_record,
        initial_winner,
        base_prompt_template,
    )
    print(f"\nDiagnóstico do Optimizer (Gemini 4 Argon):\n  {opt_meta.get('diagnosis', '')}\n")

    optimized_candidates = opt_meta.get("optimized_prompts", [])
    optimized_runs: List[Dict[str, Any]] = []

    with ThreadPoolExecutor(max_workers=4) as opt_pool:
        fut_to_meta = {}
        for idx, opt_item in enumerate(optimized_candidates, start=1):
            opt_key = opt_item.get("key", f"OPT_V{idx}")
            opt_title = opt_item.get("title", f"Prompt Otimizado v{idx}")
            raw_template = opt_item.get("template", "")
            print(f"[Otimização {idx}/{len(optimized_candidates)}] Submetendo `{best_model_id}` (MEDIUM e HIGH) com `{opt_key}`...")
            print(f"  Justificativa: {opt_item.get('rationale', '')}")
            for test_lvl in (["NATIVE_AUDIO"] if "transcribe" in best_model_id.lower() else ["MEDIUM", "HIGH"]):
                fut = opt_pool.submit(
                    evaluate_single_model_prompt_run,
                    best_model_id,
                    test_lvl,
                    opt_key,
                    opt_title,
                    raw_template,
                )
                fut_to_meta[fut] = (opt_key, test_lvl, opt_item, raw_template)

        for fut in as_completed(fut_to_meta):
            opt_key, test_lvl, opt_item, raw_template = fut_to_meta[fut]
            try:
                opt_row = fut.result()
                opt_row["is_optimized"] = True
                opt_row["optimizer_rationale"] = opt_item.get("rationale", "")
                opt_row["prompt_template"] = raw_template
                optimized_runs.append(opt_row)
                print(
                    f"  [+] Resultado `{opt_key}` [thinking={test_lvl}] -> Composite: {opt_row['composite_score']:5.2f} "
                    f"(Delta: {opt_row['composite_score'] - initial_winner['composite_score']:+5.2f}) | "
                    f"Argon Judge: {opt_row['argon_judge_score']:4.0f} | WER: {opt_row['wer_pct']:5.1f}% | "
                    f"CER: {opt_row['cer_pct']:5.1f}% | Diarização: {opt_row['speaker_role_accuracy_pct']:5.1f}% | "
                    f"Legendas 4-10s: {opt_row['subtitle_compliance_pct']:5.1f}%"
                )
            except Exception as e:
                print(f"  [ERRO ao avaliar prompt otimizado {opt_key} ({test_lvl})]: {e}")

    BENCHMARK_RESULTS = base_results_only + optimized_runs
    BENCHMARK_RESULTS.sort(key=lambda r: (-r["composite_score"], r["wer_pct"], r["latency_sec"]))

    all_model_runs = [initial_winner] + optimized_runs
    all_model_runs.sort(key=lambda r: (-r["composite_score"], r["wer_pct"]))
    final_best_for_winner = all_model_runs[0]

    print("\n" + "=" * 130)
    print(f"RESULTADO DO PROMPT OPTIMIZER PARA O MODELO GA VENCEDOR (`{best_model_id}`): ANTES vs DEPOIS")
    print("=" * 130)
    print(
        f"{'VERSÃO DO PROMPT':<38} {'THINKING':<10} {'SCORE':>8} {'DELTA':>8} {'ARGON':>8} "
        f"{'WER%':>8} {'CER%':>8} {'DIAR%':>8} {'LEG 4-10s':>10} {'LAT(s)':>8}"
    )
    print("-" * 130)
    for r in [initial_winner] + sorted(optimized_runs, key=lambda x: -x["composite_score"]):
        delta = r["composite_score"] - initial_winner["composite_score"]
        tag = " (BASELINE)" if r is initial_winner else " (OTIMIZADO)"
        print(
            f"{(r['prompt_key'] + tag):<38} {r.get('thinking_level', 'N/A'):<10} {r['composite_score']:>8.2f} {delta:>+8.2f} "
            f"{r['argon_judge_score']:>8.1f} {r['wer_pct']:>7.1f}% {r['cer_pct']:>7.1f}% "
            f"{r['speaker_role_accuracy_pct']:>7.1f}% {r['subtitle_compliance_pct']:>9.1f}% {r['latency_sec']:>7.2f}s"
        )
    print("=" * 130)

    best_prompt_txt_path = OUTPUT_DIR / "best_optimized_prompt.txt"
    best_prompt_txt_path.write_text(
        final_best_for_winner.get("prompt_template")
        or PROMPTS_REGISTRY.get(final_best_for_winner["prompt_key"], {}).get("template", ""),
        encoding="utf-8",
    )

    OPTIMIZATION_REPORT = {
        "winning_model_id": best_model_id,
        "winning_initial_thinking_level": best_thinking_lvl,
        "diagnosis": opt_meta.get("diagnosis", ""),
        "baseline_run": {k: v for k, v in initial_winner.items() if k != "segments"},
        "optimized_runs": [{k: v for k, v in r.items() if k != "segments"} for r in optimized_runs],
        "best_final_prompt_key": final_best_for_winner["prompt_key"],
        "best_final_thinking_level": final_best_for_winner.get("thinking_level"),
        "best_final_composite_score": final_best_for_winner["composite_score"],
        "score_improvement": round(
            final_best_for_winner["composite_score"] - initial_winner["composite_score"], 2
        ),
    }
    (OUTPUT_DIR / "prompt_optimization_report.json").write_text(
        json.dumps(OPTIMIZATION_REPORT, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (OUTPUT_DIR / "benchmark_results.json").write_text(
        json.dumps(
            {
                "ground_truth": ground_truth_record,
                "prompt_optimization": OPTIMIZATION_REPORT,
                "results": BENCHMARK_RESULTS,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    csv_fields = [
        "model_id",
        "thinking_level",
        "prompt_key",
        "composite_score",
        "argon_judge_score",
        "wer_pct",
        "cer_pct",
        "speaker_role_accuracy_pct",
        "mean_timestamp_drift_sec",
        "subtitle_compliance_pct",
        "segment_count",
        "avg_segment_duration_sec",
        "thinking_tokens",
        "latency_sec",
        "rtf",
        "estimated_cost_usd",
        "strengths",
        "weaknesses",
    ]
    with open(OUTPUT_DIR / "benchmark_summary.csv", "w", newline="", encoding="utf-8") as csv_f:
        writer = csv.DictWriter(csv_f, fieldnames=csv_fields)
        writer.writeheader()
        for r in BENCHMARK_RESULTS:
            writer.writerow({k: r.get(k) for k in csv_fields})

    print(f"\nPrompt otimizado salvo em: {best_prompt_txt_path}")
    print(f"Relatório de otimização salvo em: {OUTPUT_DIR / 'prompt_optimization_report.json'}")

INICIANDO PROMPT OPTIMIZER (GEMINI 4 ARGON) PARA O MODELO GA VENCEDOR: gemini-2.5-flash [thinkingLevel=HIGH] (Prompt Base: P2_VERBATIM_FORENSIC | Score Inicial: 96.23)

Diagnóstico do Optimizer (Gemini 4 Argon):
  O prompt anterior (P2_VERBATIM_FORENSIC) alcançou excelente fidelidade lexical (WER de 1.66% e CER de 0.45%) e separação de papéis processuais (100%), mas apresentou cinco gargalos estruturais na execução pelo gemini-2.5-flash: (1) Viés de OCR sobre Contexto Oral: na ausência de instrução explícita para priorizar a qualificação nominal ditada em áudio sobre os rótulos visuais do aplicativo de videoconferência, o modelo copiou os crachás de tela ('PENITENCIÁRIA DE MURIAÉ', 'CEJUSC Muriaé', 'gabriele' em minúscula) em vez de nomear 'João Vitor da Silva Andrade', 'Conciliadora CEJUSC Muriaé' e 'Gabriele'; (2) Restrição Semântica de isKeyStatement: a diretriz 4 limitava isKeyStatement a 'confissões, negativas de autoria, contradições ou decisões interlocutórias' (escopo penal/ins

In [9]:
# ==============================================================================
# 7. VISUALIZAÇÃO GRÁFICA (MATPLOTLIB) & FUNÇÃO MULTI-VÍDEOS
# ==============================================================================
try:
    import matplotlib.pyplot as plt

    if BENCHMARK_RESULTS:
        labels = [f"{r['model_id']}\n({r['prompt_key'].split('_')[0]})" for r in BENCHMARK_RESULTS]
        comp_scores = [r["composite_score"] for r in BENCHMARK_RESULTS]
        wers = [r["wer_pct"] for r in BENCHMARK_RESULTS]
        latencies = [r["latency_sec"] for r in BENCHMARK_RESULTS]

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Gráfico 1: Composite Score (0-100)
        axes[0].barh(labels[::-1], comp_scores[::-1], color="#10b981")
        axes[0].set_title("Composite Quality Score (Maior = Melhor)")
        axes[0].set_xlim(0, 105)
        axes[0].set_xlabel("Score (0–100)")

        # Gráfico 2: Word Error Rate (WER %)
        axes[1].barh(labels[::-1], wers[::-1], color="#f59e0b")
        axes[1].set_title("Word Error Rate vs Gemini 4 Argon GT (Menor = Melhor)")
        axes[1].set_xlabel("WER (%)")

        # Gráfico 3: Latência vs Qualidade (Fronteira de Pareto)
        axes[2].scatter(latencies, comp_scores, s=110, c="#ef4444", alpha=0.85)
        for lbl, x, y in zip(labels, latencies, comp_scores):
            axes[2].annotate(lbl.replace("\n", " "), (x, y), fontsize=8, xytext=(5, 4), textcoords="offset points")
        axes[2].set_title("Fronteira de Pareto: Latência (s) vs Qualidade")
        axes[2].set_xlabel("Latência de Transcrição (s)")
        axes[2].set_ylabel("Composite Score (0–100)")
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        chart_path = OUTPUT_DIR / "benchmark_comparison_chart.png"
        plt.savefig(chart_path, dpi=150)
        print(f"Gráfico comparativo salvo em: {chart_path}")
        plt.show()
except ImportError:
    print("Dica: instale `matplotlib` (`pip install matplotlib`) caso queira renderizar os gráficos PNG.")

Dica: instale `matplotlib` (`pip install matplotlib`) caso queira renderizar os gráficos PNG.
